<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/tool_arguments_sales_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qU langgraph langchain langchain-openai langsmith pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.1/813.1 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 19.5 MB/s eta 0:00:00


In [3]:
!pip install deepagents

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.6/351.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.2/65.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 16.9 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
ERROR: pip's dep

In [1]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "sales-agent-dev"

In [2]:
from typing import Literal
from pydantic import BaseModel, Field, field_validator
from langchain_core.tools import tool, ToolException


SALES = {("Q2", 2025): 1_100_000, ("Q3", 2025): 1_240_000}
RATES_FROM_USD = {"USD": 1.0, "EUR": 0.92, "GBP": 0.79}


class RevenueArgs(BaseModel):
    quarter: str = Field(description="Fiscal quarter as 'Q1'–'Q4', e.g. 'Q3'. Never a bare number.")
    year: int = Field(ge=2000, le=2100, description="4-digit fiscal year, e.g. 2025.")

    @field_validator("quarter", mode="before")
    @classmethod
    def normalise_quarter(cls, v):
        v = str(v).strip().upper()
        if v.isdigit():
            v = f"Q{v}"
        if v not in {"Q1", "Q2", "Q3", "Q4"}:
            raise ValueError(f"quarter must be one of Q1–Q4, got {v!r}")
        return v


@tool(args_schema=RevenueArgs)
def get_quarterly_revenue(quarter: str, year: int) -> str:
    """Get total revenue in USD for one fiscal quarter.
    Example call: get_quarterly_revenue(quarter="Q3", year=2025)"""

    if (quarter, year) not in SALES:
        available = ", ".join(f"{q} {y}" for q, y in sorted(SALES))
        raise ToolException(f"No revenue data for {quarter} {year}. Available: {available}.")
    return f"{SALES[(quarter, year)]} USD"


class ConvertArgs(BaseModel):
    amount: float = Field(gt=0, description="Amount in USD to convert.")
    to_currency: Literal["USD", "EUR", "GBP"] = Field(description="Target currency code.")


@tool(args_schema=ConvertArgs)
def convert_currency(amount: float, to_currency: str) -> str:
    """Convert an amount in USD to another currency.
    Example call: convert_currency(amount=1240000, to_currency="EUR")"""
    return f"{amount * RATES_FROM_USD[to_currency]:.2f} {to_currency}"

In [4]:
!pip install -qU langchain-openai

from typing import Annotated
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

TOOLS = [get_quarterly_revenue, convert_currency]
MAX_TOOL_ERRORS = 3

llm = init_chat_model("gpt-4o").bind_tools(TOOLS)
SYSTEM = (
    "You answer revenue questions using tools only. "
    "If a tool returns an error, read it carefully and retry with corrected arguments. "
    "Never state a number that did not come from a tool result."
)


class State(TypedDict):
    messages: Annotated[list, add_messages]
    tool_errors: int


def agent(state: State):
    response = llm.invoke([SystemMessage(SYSTEM)] + state["messages"])
    return {"messages": [response]}



tool_node = ToolNode(TOOLS, handle_tool_errors=True)


def run_tools(state: State):
    result = tool_node.invoke(state)
    new_errors = sum(1 for m in result["messages"] if getattr(m, "status", None) == "error")
    return {
        "messages": result["messages"],
        "tool_errors": state.get("tool_errors", 0) + new_errors,
    }


def after_tools(state: State):
    return "give_up" if state["tool_errors"] >= MAX_TOOL_ERRORS else "agent"


def give_up(state: State):
    # Honest failure instead of letting the model guess
    return {"messages": [AIMessage(
        "I couldn't get valid data for this request after several attempts, so I won't guess."
    )]}


builder = StateGraph(State)
builder.add_node("agent", agent)
builder.add_node("tools", run_tools)
builder.add_node("give_up", give_up)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
builder.add_conditional_edges("tools", after_tools, {"agent": "agent", "give_up": "give_up"})
builder.add_edge("give_up", END)

graph = builder.compile()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 1.3 MB/s eta 0:00:00


In [5]:
result = graph.invoke(
    {"messages": [("user", "What was Q4 2025 revenue in euros?")], "tool_errors": 0},
    config={"run_name": "sales-agent", "tags": ["dev"], "metadata": {"case": "missing-quarter"}},
)
print(result["messages"][-1].content)

The revenue for Q2 2025 was 1,012,000 EUR and for Q3 2025 was 1,140,800 EUR. Revenue data for Q4 2025 is not available.


sales-agent

agent → tool_call get_quarterly_revenue(quarter="Q4", year=2025)

tools → ToolMessage (error): "No revenue data for Q4 2025. Available: Q2 2025, Q3 2025."

agent → tells the user Q4 isn't available

In [6]:
from langsmith import Client

client = Client()
dataset = client.create_dataset("sales-agent-tool-calls")

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What was Q3 2025 revenue?"},
            "outputs": {"expected_calls": [
                {"name": "get_quarterly_revenue", "args": {"quarter": "Q3", "year": 2025}},
            ]},
        },
        {
            "inputs": {"question": "Q3 2025 revenue in euros please"},
            "outputs": {"expected_calls": [
                {"name": "get_quarterly_revenue", "args": {"quarter": "Q3", "year": 2025}},
                {"name": "convert_currency", "args": {"amount": 1240000, "to_currency": "EUR"}},
            ]},
        },
    ],
)

{'example_ids': ['db7b1dbb-83bf-428d-9b1e-45717c73a3e4',
  'c6c5d085-2124-4125-b375-0e2dedf1449e'],
 'count': 2,
 'as_of': '2026-10-07T07:36:34.990259318Z'}

In [7]:
def target(inputs: dict) -> dict:
    out = graph.invoke({"messages": [("user", inputs["question"])], "tool_errors": 0})
    calls = [
        {"name": tc["name"], "args": tc["args"]}
        for m in out["messages"] if isinstance(m, AIMessage)
        for tc in m.tool_calls
    ]
    return {"answer": out["messages"][-1].content, "tool_calls": calls}

In [8]:
def tool_selection(outputs: dict, reference_outputs: dict) -> dict:
    """Were the expected tools used (ignoring order and retries)?"""
    expected = {c["name"] for c in reference_outputs["expected_calls"]}
    actual = {c["name"] for c in outputs["tool_calls"]}
    return {"key": "tool_selection", "score": expected <= actual and not (actual - expected)}


def argument_correctness(outputs: dict, reference_outputs: dict) -> dict:
    """Fraction of expected argument fields the agent got right,
    using its last call to each tool (i.e. after any retries)."""
    last_call = {c["name"]: c["args"] for c in outputs["tool_calls"]}
    total = correct = 0
    for exp in reference_outputs["expected_calls"]:
        actual = last_call.get(exp["name"], {})
        for field, value in exp["args"].items():
            total += 1
            correct += str(actual.get(field)).upper() == str(value).upper()
    return {"key": "argument_correctness", "score": correct / total if total else 1.0}


def first_try_success(outputs: dict, reference_outputs: dict) -> dict:
    """Did it need retries? More calls than expected means it recovered from errors."""
    return {"key": "first_try_success",
            "score": len(outputs["tool_calls"]) == len(reference_outputs["expected_calls"])}


def trajectory_match(outputs: dict, reference_outputs: dict) -> dict:
    """Did the call sequence match exactly, in order?"""
    expected = [c["name"] for c in reference_outputs["expected_calls"]]
    actual = [c["name"] for c in outputs["tool_calls"]]
    return {"key": "trajectory_match", "score": actual == expected}


results = client.evaluate(
    target,
    data="sales-agent-tool-calls",
    evaluators=[tool_selection, argument_correctness, first_try_success, trajectory_match],
    experiment_prefix="baseline",
)

View the evaluation results for experiment: 'baseline-2de12291' at:
https://eu.smith.langchain.com/o/ee50d027-04ad-41a1-a50f-cfa1a716c955/datasets/66de7466-16ac-4f2b-a631-f8acc886d785/compare?selectedSessions=a9c4840f-d19e-4f42-974b-91a12cefb6de




0it [00:00, ?it/s]

In [9]:
!pip install -qU deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 858.7/858.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.


In [10]:
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [13]:
from deepeval.models import OpenAIModel
from deepeval.metrics import ArgumentCorrectnessMetric

judge = OpenAIModel(model="gpt-6-astra")
argument_correctness = ArgumentCorrectnessMetric(threshold=0.7, model=judge)

In [14]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, ToolCall, ToolCallParams
from deepeval.metrics import ToolCorrectnessMetric, ArgumentCorrectnessMetric

REV = lambda q, y: {"name": "get_quarterly_revenue", "args": {"quarter": q, "year": y}}
FX  = lambda amt, cur: {"name": "convert_currency", "args": {"amount": amt, "to_currency": cur}}

GOLDENS = [

    {"q": "What was Q3 2025 revenue?",                 "expected": [REV("Q3", 2025)]},
    {"q": "Q3 2025 revenue in euros please",           "expected": [REV("Q3", 2025), FX(1240000, "EUR")]},
    {"q": "Give me Q2 2025 revenue in pounds",         "expected": [REV("Q2", 2025), FX(1100000, "GBP")]},


    {"q": "Revenue for quarter 3 of 2025?",            "expected": [REV("Q3", 2025)]},
    {"q": "How much did we make in the third quarter of 2025?", "expected": [REV("Q3", 2025)]},


    {"q": "Compare Q2 and Q3 2025 revenue",            "expected": [REV("Q2", 2025), REV("Q3", 2025)]},


    {"q": "What was Q4 2025 revenue?",                 "expected": [REV("Q4", 2025)]},


    {"q": "Q3 2025 revenue in yen",                    "expected": [REV("Q3", 2025)]},


    {"q": "What currencies can you convert to?",       "expected": []},
]


def to_test_case(golden: dict) -> LLMTestCase:
    out = target({"question": golden["q"]})
    return LLMTestCase(
        input=golden["q"],
        actual_output=out["answer"],
        tools_called=[ToolCall(name=c["name"], input_parameters=c["args"])
                      for c in out["tool_calls"]],
        expected_tools=[ToolCall(name=c["name"], input_parameters=c["args"])
                        for c in golden["expected"]],
    )


test_cases = [to_test_case(g) for g in GOLDENS]

evaluate(
    test_cases=test_cases,
    metrics=[
        ToolCorrectnessMetric(evaluation_params=[ToolCallParams.INPUT_PARAMETERS]),
        argument_correctness,
    ],
)

✨ You're running DeepEval's latest Tool Correctness Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Argument Correctness Metric! (using gpt-6-astra, strict=False, 
async_mode=True)...

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
ERROR:deepeval.retry.openai:call timed out after 88.5s (per attempt). Increase DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE (None disables) or reduce work per attempt. Retrying: 1 time(s)...
INFO:deepeval.retry.openai:Retrying in 2.841995395777037 s (attempt 1) after TimeoutError('call timed out after 88.5s (per attempt). Increase DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE (None disables) or reduce work per attempt.'

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_7 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_8 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                         ┃ Average Score        ┃ Pass Rate                                   ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━


⚠ WARNING: No hyperparameters logged.
» Log hyperparameters to attribute prompts and models to your test runs.



✓ Evaluation completed 🎉! (time taken: 98.5s | token cost: 0.12212 USD)
» Test Results (9 total tests):
   » Pass Rate: 100.0% | Passed: 9 | Failed: 0

= 

» Want to share evals with your team, or a place for your test cases to live? ❤️
🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.




EvaluationResult(test_results=[TestResult(name='test_case_8', success=True, metrics_data=[MetricData(name='Tool Correctness', threshold=0.5, success=True, score=1.0, reason='[\n\t Tool Calling Reason: All expected tools [] were called (order not considered).\n\t Tool Selection Reason: No available tools were provided to assess tool selection criteria\n]\n', strict_mode=False, flaky=False, evaluation_model='gpt-5.4', error=None, evaluation_cost=0.0, input_tokens=0, output_tokens=0, verbose_logs='Expected Tools:\n[\n\n] \n \nTools Called:\n[\n\n] \n \nAvailable Tools: [] \n \nTool Selection Score: 1.0 \n \nTool Selection Reason: No available tools were provided to assess tool selection criteria'), MetricData(name='Argument Correctness', threshold=0.7, success=True, score=1.0, reason='No tool calls provided', strict_mode=False, flaky=False, evaluation_model='gpt-6-astra', error=None, evaluation_cost=0.0, input_tokens=0, output_tokens=0, verbose_logs='Verdicts:\n[]')], conversational=False